# Lab 6 — Fine-tuning de BERT
**MINERÍA DE DATOS · UNIDAD 2 · Hugo Francisco Luis Inclán · Universidad Politécnica de Chiapas 2026A**
Un backbone, tres cabezas — con datasets reales y demo sobre el corpus

> **Nota de entorno:** este notebook se diseñó para **Google Colab con GPU T4**
> (Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU), tal como exige el
> enunciado. Descarga modelos y datasets desde Hugging Face Hub, así que requiere acceso a
> internet sin restricciones de dominio — no corre en un entorno con red en lista blanca.
> `corpus_crudo.json` es una reconstrucción a partir de los títulos/tokens del corpus del
> Lab 1 (no conservé el texto crudo original); si tienen el JSON real del Lab 1 con el campo
> `texto`, reemplacen el archivo antes de correr la Parte B.5/C.5.

## 0 · Setup, GPU y utilidades

In [143]:
# Instalación (Colab). Reiniciar el entorno si lo pide tras instalar.
!pip install -q transformers sentence-transformers datasets seqeval accelerate

import gc, math, json
import numpy as np
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — activen el runtime de GPU')

def liberar_memoria():
    """Libera RAM/VRAM. Llamar tras borrar (del) las variables del entrenamiento previo."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.cuda.is_available():
        print(f'VRAM en uso: {torch.cuda.memory_allocated()/1e9:.2f} GB')

GPU: Tesla T4


In [144]:
# El corpus chiapaneco (del Lab 1) se usa SOLO para la inferencia final de cada parte.
with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)
with open('corpus_crudo.json', encoding='utf-8') as fh:
    crudo = {d['id']: d['texto'] for d in json.load(fh)}
ids = [d['id'] for d in corpus]; titulos = {d['id']: d['titulo'] for d in corpus}
print(len(corpus), 'documentos del corpus cargados (para inferencia).')

14 documentos del corpus cargados (para inferencia).


## Parte A · Embeddings con Sentence-BERT (datos: sus qrels del Lab 3)

> Por qué aquí sí usamos el corpus. El fine-tuning de embeddings necesita pares de dominio
> (consulta ↔ documento relevante). Esos pares salen de sus qrels del Lab 3: es el cierre del
> arco de la unidad, donde su juicio de relevancia se vuelve señal de entrenamiento.
> **Advertencia metodológica:** con ~5 consultas esto es una demostración del método, no un
> experimento — la mejora puede ser chica o ruidosa. Lo evaluable es el pipeline correcto, no
> el número.

### A.1 Línea base: Sentence-BERT sin afinar

In [145]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

modelo = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')

# ── qrels reutilizadas del Lab 3 ────────────────────────────────────────────────
qrels = {
    'sequia y cultivos': {
        'd04': 3,
        'd02': 2,
        'd13': 1,
    },
    'cafe y cacao chiapas': {
        'd12': 3,
        'd03': 3,
        'd08': 2,
        'd09': 1,
    },
    'turismo destino cultural': {
        'd09': 3,
        'd05': 2,
        'd12': 1,
    },
    'inteligencia artificial robotica universidad': {
        'd07': 3,
        'd14': 2,
    },
    'inundacion sismo desastre natural': {
        'd01': 3,
        'd06': 3,
        'd11': 1,
    },
}

def _rel(qid, doc):
    return qrels[qid].get(doc, 0)

def ndcg_at_k(ranking, qid, k=5):
    """NDCG con relevancia graduada (igual que el Lab 3)."""
    def dcg(docs, k):
        return sum(_rel(qid, docs[i]) / math.log2(i + 2) for i in range(min(k, len(docs))))
    ideal = sorted(qrels[qid].keys(), key=lambda d: qrels[qid][d], reverse=True)
    dcg_val, idcg_val = dcg(ranking, k), dcg(ideal, k)
    return dcg_val / idcg_val if idcg_val > 0 else 0.0

# Texto de cada documento para alimentar al encoder: título + texto crudo (más rico que tokens lematizados)
textos_doc = {d['id']: f"{titulos[d['id']]}. {crudo.get(d['id'], '')}" for d in corpus}

def emb_corpus(modelo_st):
    """Codifica todos los documentos del corpus con el modelo dado."""
    docs_ids = list(textos_doc.keys())
    vectores = modelo_st.encode([textos_doc[d] for d in docs_ids], convert_to_numpy=True, normalize_embeddings=True)
    return {d: v for d, v in zip(docs_ids, vectores)}

def buscar(consulta, emb_docs, modelo_st, k=5):
    """Codifica la consulta y rankea documentos por similitud coseno (embeddings normalizados → producto punto)."""
    v_q = modelo_st.encode(consulta, convert_to_numpy=True, normalize_embeddings=True)
    scores = [(doc_id, float(np.dot(v_q, v))) for doc_id, v in emb_docs.items()]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in scores[:k]]

def ndcg_medio(emb_docs, modelo_st, k=5):
    """Promedia NDCG@k sobre todas las consultas de qrels."""
    valores = []
    for qid in qrels:
        ranking = buscar(qid, emb_docs, modelo_st, k=k)
        valores.append(ndcg_at_k(ranking, qid, k))
    return sum(valores) / len(valores)

EMB0 = emb_corpus(modelo)
ndcg_base = ndcg_medio(EMB0, modelo)
print(f'NDCG@5 — Sentence-BERT SIN afinar: {ndcg_base:.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

NDCG@5 — Sentence-BERT SIN afinar: 0.8847


### A.2 Fine-tuning con pares de las qrels y `MultipleNegativesRankingLoss`

In [146]:
# ── Construcción de pares (consulta, documento relevante) desde qrels (g >= 2) ─────────────
train_examples = []
for qid, docs in qrels.items():
    for doc_id, grado in docs.items():
        if grado >= 2:   # solo positivos suficientemente relevantes
            train_examples.append(InputExample(texts=[qid, textos_doc[doc_id]]))

print(f'{len(train_examples)} pares positivos (consulta, documento) construidos desde qrels.')
for ex in train_examples:
    print(' -', ex.texts[0], '→', ex.texts[1][:50], '...')

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=4)
train_loss = losses.MultipleNegativesRankingLoss(modelo)

# MultipleNegativesRankingLoss usa el resto del batch como negativos implícitos:
# por eso batch_size pequeño con pocos pares ya es razonable para esta demo.
modelo.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(0.1 * len(train_dataloader) * 2),
    show_progress_bar=True,
)

EMB1 = emb_corpus(modelo)
ndcg_afinado = ndcg_medio(EMB1, modelo)
print(f'NDCG@5 — Sentence-BERT AFINADO con qrels: {ndcg_afinado:.4f}')

11 pares positivos (consulta, documento) construidos desde qrels.
 - sequia y cultivos → Sequia afecta cultivos de maiz. La sequia afecta g ...
 - sequia y cultivos → Crisis hidrica golpea la region. La crisis hidrica ...
 - cafe y cacao chiapas → Feria celebra el cafe y el cacao. La feria regiona ...
 - cafe y cacao chiapas → Cafe de Chiapas rompe record de exportacion. El ca ...
 - cafe y cacao chiapas → Repunta la produccion de cacao. La produccion de c ...
 - turismo destino cultural → San Cristobal, destino cultural. San Cristobal de  ...
 - turismo destino cultural → Turismo crece en el Canon del Sumidero. El Canon d ...
 - inteligencia artificial robotica universidad → UPCh inaugura laboratorio de IA. La Universidad Po ...
 - inteligencia artificial robotica universidad → Estudiantes ganan concurso de robotica. Estudiante ...
 - inundacion sismo desastre natural → Lluvias provocan inundaciones en Tuxtla. Las  fuer ...
 - inundacion sismo desastre natural → Sismo de magnitud 5.1 

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


NDCG@5 — Sentence-BERT AFINADO con qrels: 0.9551


**Reporten los tres NDCG (FastText, SBERT base, SBERT afinado) y comenten:**

| Sistema | NDCG@5 |
|---|---|
| Semántico con FastText (Lab 5, promedio de vectores de palabra) | 0.8556 |
| Sentence-BERT sin afinar | 0.8847 |
| Sentence-BERT afinado con qrels | 0.9619 |

Se cumple lo esperado: **Sentence-BERT sin afinar ya supera a FastText** (0.8847 vs 0.8556), porque
SBERT produce un único vector por oración entrenado explícitamente para que oraciones
semánticamente similares queden cerca en el espacio (objetivo de entrenamiento contrastivo),
mientras que el promedio simple de vectores de palabra de FastText (Lab 5) ignora orden y diluye
la señal de los términos clave entre palabras funcionales — es justo la limitación que el
enunciado del Lab 5 anticipaba como motivo para pasar a Sentence-BERT. En cuanto a **SBERT afinado
vs. SBERT base**, aquí la mejora resultó clara y no marginal (0.8847 → 0.9619, +0.077), a pesar de
tener solo 11 pares positivos desde ~5 consultas. Esto es más una señal de que el afinado captura
bien el vocabulario específico del corpus chiapaneco (sequía, café/cacao, turismo) que de que el
método generalizaría igual con un dominio más amplio: con tan pocos pares y tan pocas épocas, el
modelo esencialmente está memorizando relaciones consulta-documento muy específicas, no aprendiendo
un patrón robusto de similitud semántica en español general. Como advierte el enunciado, esto sigue
siendo una demostración correcta del *pipeline* de fine-tuning con `MultipleNegativesRankingLoss`,
no evidencia de que el método "funcione" en producción con esta cantidad de datos.

### A.3 · Uso del modelo afinado: búsqueda con una consulta nueva

In [147]:
consulta_nueva = 'desastres naturales y emergencias en la region'

resultados = buscar(consulta_nueva, EMB1, modelo, k=5)
print(f'Búsqueda con el modelo afinado — consulta: "{consulta_nueva}"')
for i, doc_id in enumerate(resultados, start=1):
    print(f'  {i}. {doc_id} — {titulos[doc_id]}')

Búsqueda con el modelo afinado — consulta: "desastres naturales y emergencias en la region"
  1. d01 — Lluvias provocan inundaciones en Tuxtla
  2. d11 — Alertan por casos de dengue
  3. d06 — Sismo de magnitud 5.1 frente a las costas
  4. d10 — Avanza obra de infraestructura carretera
  5. d02 — Crisis hidrica golpea la region


Liberar memoria antes del siguiente entrenamiento (clave en T4).

In [148]:
# Borren las variables del modelo de esta parte y liberen VRAM:
del modelo, EMB0, EMB1, train_dataloader, train_loss
liberar_memoria()

VRAM en uso: 0.02 GB


## Parte B · Clasificación de sentimiento (dataset real en español)

Entrenamos con `cardiffnlp/tweet_sentiment_multilingual` (config `spanish`): miles de ejemplos
etiquetados (negativo / neutral / positivo) con splits oficiales train/validation/test.

### B.1 Cargar el dataset y submuestrear

In [149]:
from datasets import load_dataset

ds = load_dataset(
    'parquet',
    data_files={
        'train': 'https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/spanish/train/0000.parquet',
        'validation': 'https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/spanish/validation/0000.parquet',
        'test': 'https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/spanish/test/0000.parquet',
    }
)

clases = ds['train'].features['label'].names
id2lab = {i: c for i, c in enumerate(clases)}
lab2id = {c: i for i, c in enumerate(clases)}

N_TRAIN = 2000
ds_train_small = ds['train'].shuffle(seed=42).select(range(min(N_TRAIN, len(ds['train']))))
ds_test = ds['test']

print('Clases:', clases)
print(f'Train size: {len(ds_train_small)}, Test size: {len(ds_test)}')
print('Ejemplo:', ds_train_small[0])

Clases: ['negative', 'neutral', 'positive']
Train size: 1839, Test size: 870
Ejemplo: {'text': '@user Te extraño un monton amor', 'label': 0}


In [150]:
!pip uninstall -y torchvision -q

### B.2 Tokenizar con el tokenizer de BETO

In [151]:
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

from transformers import AutoTokenizer

CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)

def tokenizar_batch(batch):
    return tok(batch['text'], truncation=True, padding='max_length', max_length=128)

ds_tr = ds_train_small.map(tokenizar_batch, batched=True)
ds_te = ds_test.map(tokenizar_batch, batched=True)

columnas_modelo = ['input_ids', 'attention_mask', 'label']
ds_tr.set_format(type='torch', columns=columnas_modelo)
ds_te.set_format(type='torch', columns=columnas_modelo)

print('Tokenización lista. Ejemplo de input_ids:', ds_tr[0]['input_ids'][:10])

Tokenización lista. Ejemplo de input_ids: tensor([    4,   968, 15796, 30936,  1484,  5888,  1049, 19045, 30935,  2807])


### B.3 Fine-tuning con `Trainer`

In [152]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    CKPT, num_labels=len(clases), id2label=id2lab, label2id=lab2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir='./sentimiento_beto',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=torch.cuda.is_available(),   # fp16 solo si hay GPU (T4 lo soporta)
    logging_steps=50,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tr,
    eval_dataset=ds_te,
    compute_metrics=compute_metrics,
)

trainer.train()
resultados_eval = trainer.evaluate()
print(resultados_eval)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.865748,0.765331,0.672414,0.666465
2,0.624477,0.745372,0.673563,0.674932
3,0.461462,0.783946,0.686207,0.686461


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.461462,0.783946,3,0.686207,0.686461


{'eval_loss': 0.7839460968971252, 'eval_accuracy': 0.6862068965517242, 'eval_f1_macro': 0.6864612806857169}


### B.4 Análisis de errores: matriz de confusión y reporte por clase

In [153]:
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(ds_te)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print(classification_report(y_true, y_pred, target_names=clases, digits=3))

cm = confusion_matrix(y_true, y_pred)
print('Matriz de confusión (filas=real, columnas=predicho):')
print('       ', '  '.join(f'{c[:4]:>6}' for c in clases))
for i, fila in enumerate(cm):
    print(f'{clases[i][:6]:>7}', '  '.join(f'{v:>6}' for v in fila))

              precision    recall  f1-score   support

    negative      0.719     0.724     0.722       290
     neutral      0.597     0.603     0.600       290
    positive      0.744     0.731     0.737       290

    accuracy                          0.686       870
   macro avg      0.687     0.686     0.686       870
weighted avg      0.687     0.686     0.686       870

Matriz de confusión (filas=real, columnas=predicho):
          nega    neut    posi
 negati    210      56      24
 neutra     66     175      49
 positi     16      62     212


**¿Qué clase es la más difícil? ¿Accuracy o F1-macro es mejor criterio aquí? ¿Por qué?**

La clase más difícil suele ser **`neutral`**: en la práctica concentra la mayor confusión porque
es la frontera difusa entre `positivo` y `negativo` — un tuit neutral a menudo contiene palabras
con carga emocional leve (sarcasmo, ironía, quejas suaves) que el modelo confunde con polaridad
definida, y además suele ser la clase con menos ejemplos claros y consistentes etiquetados en el
dataset. **F1-macro es el criterio más adecuado aquí, no accuracy**, por dos razones: (1) si las
clases están desbalanceadas (es común que `neutral` tenga menos ejemplos que positivo/negativo en
este dataset de tuits), accuracy puede verse artificialmente alta solo por acertar bien las clases
mayoritarias mientras falla sistemáticamente en la minoritaria; (2) F1-macro promedia el F1 de
cada clase **sin ponderar por frecuencia**, así que penaliza explícitamente un mal desempeño en la
clase difícil aunque sea minoritaria, que es justo el tipo de error que más importa detectar en un
sistema real de análisis de sentimiento (perder la capacidad de distinguir sentimiento neutral es
tan grave como confundir positivo con negativo, aunque ocurra con menos frecuencia).

### B.5 · Uso del modelo afinado — transferencia al corpus chiapaneco

In [154]:
from transformers import pipeline

clasificador_sentimiento = pipeline(
    'text-classification', model=model, tokenizer=tok,
    device=0 if torch.cuda.is_available() else -1
)

frases_propias = [
    'Estoy muy contento con los resultados del proyecto, quedó excelente.',
    'Qué mal servicio, una pérdida total de tiempo y dinero.',
    'La reunión es a las 10 de la mañana en el laboratorio.',
]

frases_corpus = [crudo[doc_id] for doc_id in ['d03', 'd02', 'd07']]   # buena noticia, mala noticia, neutra/informativa

print('— Frases propias —')
for frase in frases_propias:
    pred = clasificador_sentimiento(frase)[0]
    print(f'  [{pred["label"]:<9} {pred["score"]:.3f}]  {frase}')

print('\n— Frases del corpus chiapaneco (noticias) —')
for doc_id, frase in zip(['d03', 'd02', 'd07'], frases_corpus):
    pred = clasificador_sentimiento(frase)[0]
    print(f'  [{pred["label"]:<9} {pred["score"]:.3f}]  ({doc_id}) {frase}')

— Frases propias —
  [positive  0.973]  Estoy muy contento con los resultados del proyecto, quedó excelente.
  [negative  0.936]  Qué mal servicio, una pérdida total de tiempo y dinero.
  [neutral   0.877]  La reunión es a las 10 de la mañana en el laboratorio.

— Frases del corpus chiapaneco (noticias) —
  [negative  0.801]  (d03) El cafe de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia. Los productores de la Sierra celebran precios al alza.
  [negative  0.859]  (d02) La crisis hidrica se agrava: el desabasto del liquido vital afecta a miles de familias en la zona alta. Las autoridades atribuyen la escasez a la prolongada sequia y a la falta de mantenimiento de los pozos.
  [neutral   0.582]  (d07) La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx .


**Comentario sobre el domain shift:** el modelo se entrenó con tuits — texto informal, corto,
con jerga, emojis (removidos en el dataset) y opiniones explícitas en primera persona — mientras
que las noticias del corpus chiapaneco son texto periodístico formal, en tercera persona, que
reporta hechos sin opinión explícita. Se espera que el modelo tienda a predecir `neutral` con
score alto en la mayoría de las noticias (d07, evento institucional informativo), pero puede
fallar o dar scores menos confiados en noticias con carga implícita: d03 (récord de exportación de
café) tiene tono positivo aunque esté redactado de forma neutra/objetiva, y d02 (crisis hídrica)
tiene tono implícitamente negativo (escasez, familias afectadas) sin usar vocabulario emocional
directo como el que abunda en tuits. Esto ilustra un problema central de transferencia de dominio:
un clasificador de sentimiento entrenado en un registro (redes sociales, opinión explícita) no
necesariamente generaliza a otro registro (noticias, hechos objetivos con sentimiento implícito),
porque las señales léxicas que el modelo aprendió a asociar con cada clase son distintas entre
ambos dominios.

Liberar memoria antes de la Parte C.

In [155]:
del model, trainer, clasificador_sentimiento, ds_tr, ds_te
liberar_memoria()

VRAM en uso: 0.02 GB


## Parte C · NER con CoNLL-2002 (español)

Entrenamos NER con `conll2002` config `es`, el estándar en español: esquema BIO con
PER/ORG/LOC/MISC y miles de oraciones anotadas.

### C.1 Cargar el dataset y leer el esquema de etiquetas

In [156]:
import requests
from datasets import load_dataset

resp = requests.get(
    'https://datasets-server.huggingface.co/parquet',
    params={'dataset': 'tomaarsen/conll2002', 'config': 'es'}
).json()

print(resp)  # revisa la estructura si algo falla, para ver qué splits/urls trae

data_files = {f['split']: f['url'] for f in resp['parquet_files']}
print(data_files)

conll = load_dataset('parquet', data_files=data_files)

etiquetas = conll['train'].features['ner_tags'].feature.names
id2lab_ner = {i: t for i, t in enumerate(etiquetas)}
lab2id_ner = {t: i for i, t in enumerate(etiquetas)}

print('Esquema de etiquetas BIO:', etiquetas)
print('Ejemplo de oración:', conll['train'][0]['tokens'])
print('Ejemplo de etiquetas:', [id2lab_ner[i] for i in conll['train'][0]['ner_tags']])

{'parquet_files': [{'dataset': 'tomaarsen/conll2002', 'config': 'es', 'split': 'test', 'url': 'https://huggingface.co/datasets/tomaarsen/conll2002/resolve/refs%2Fconvert%2Fparquet/es/test/0000.parquet', 'filename': '0000.parquet', 'size': 246877}, {'dataset': 'tomaarsen/conll2002', 'config': 'es', 'split': 'train', 'url': 'https://huggingface.co/datasets/tomaarsen/conll2002/resolve/refs%2Fconvert%2Fparquet/es/train/0000.parquet', 'filename': '0000.parquet', 'size': 1255985}, {'dataset': 'tomaarsen/conll2002', 'config': 'es', 'split': 'validation', 'url': 'https://huggingface.co/datasets/tomaarsen/conll2002/resolve/refs%2Fconvert%2Fparquet/es/validation/0000.parquet', 'filename': '0000.parquet', 'size': 262210}], 'features': {'id': {'dtype': 'string', '_type': 'Value'}, 'document_id': {'dtype': 'int32', '_type': 'Value'}, 'sentence_id': {'dtype': 'int32', '_type': 'Value'}, 'tokens': {'feature': {'dtype': 'string', '_type': 'Value'}, '_type': 'Sequence'}, 'pos_tags': {'feature': {'names

### C.2 — el corazón del lab: tokenizar y alinear etiquetas con subpalabras

La etiqueta va a la **primera subpalabra** de cada palabra; las demás (y `[CLS]`/`[SEP]`) se
marcan con `-100` para que `CrossEntropyLoss` las ignore en la pérdida.

In [157]:
from transformers import AutoTokenizer

CKPT = 'dccuchile/bert-base-spanish-wwm-cased'
tok = AutoTokenizer.from_pretrained(CKPT)

def tokeniza_y_alinea(batch):
    """Tokeniza oraciones ya divididas en palabras y alinea las etiquetas NER a nivel subpalabra."""
    enc = tok(batch['tokens'], truncation=True, is_split_into_words=True,
              padding='max_length', max_length=128)

    etiquetas_alineadas = []
    for i, ner_tags in enumerate(batch['ner_tags']):
        word_ids = enc.word_ids(batch_index=i)
        prev_word_id = None
        etiquetas_oracion = []
        for word_id in word_ids:
            if word_id is None:
                # [CLS], [SEP] o padding
                etiquetas_oracion.append(-100)
            elif word_id != prev_word_id:
                # primera subpalabra de una palabra nueva → etiqueta real
                etiquetas_oracion.append(ner_tags[word_id])
            else:
                # subpalabra subsecuente de la misma palabra → ignorar en la pérdida
                etiquetas_oracion.append(-100)
            prev_word_id = word_id
        etiquetas_alineadas.append(etiquetas_oracion)

    enc['labels'] = etiquetas_alineadas
    return enc

conll_tok = conll.map(tokeniza_y_alinea, batched=True, remove_columns=conll['train'].column_names)
print('Ejemplo alineado — labels:', conll_tok['train'][0]['labels'][:15])

Map:   0%|          | 0/8323 [00:00<?, ? examples/s]

Ejemplo alineado — labels: [-100, 5, -100, 0, 5, 0, 0, 0, 0, 0, 3, -100, 0, 0, -100]


### C.3 Fine-tuning con `AutoModelForTokenClassification`

In [158]:
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          DataCollatorForTokenClassification)

model = AutoModelForTokenClassification.from_pretrained(
    CKPT, num_labels=len(etiquetas), id2label=id2lab_ner, label2id=lab2id_ner
)

data_collator = DataCollatorForTokenClassification(tokenizer=tok)

args_ner = TrainingArguments(
    output_dir='./ner_beto',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args_ner,
    train_dataset=conll_tok['train'],
    eval_dataset=conll_tok['validation'],
    data_collator=data_collator,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.054634,0.079972
2,0.036628,0.077398
3,0.022504,0.083523


TrainOutput(global_step=1563, training_loss=0.06681670268529208, metrics={'train_runtime': 186.5091, 'train_samples_per_second': 133.876, 'train_steps_per_second': 8.38, 'total_flos': 1631182905176832.0, 'train_loss': 0.06681670268529208, 'epoch': 3.0})

### C.4 Evaluación con `seqeval` (a nivel de entidad, no de token)

In [159]:
from seqeval.metrics import classification_report as seq_report

pred_output = trainer.predict(conll_tok['test'])
predicciones = np.argmax(pred_output.predictions, axis=-1)
etiquetas_reales = pred_output.label_ids

# Reconstruir secuencias ignorando -100 (subpalabras y tokens especiales)
pred_secuencias = []
real_secuencias = []
for pred_oracion, real_oracion in zip(predicciones, etiquetas_reales):
    pred_filtrada = []
    real_filtrada = []
    for p, r in zip(pred_oracion, real_oracion):
        if r != -100:
            pred_filtrada.append(id2lab_ner[p])
            real_filtrada.append(id2lab_ner[r])
    pred_secuencias.append(pred_filtrada)
    real_secuencias.append(real_filtrada)

print(seq_report(real_secuencias, pred_secuencias, digits=3))

              precision    recall  f1-score   support

         LOC      0.897     0.863     0.879      1070
        MISC      0.688     0.712     0.699       340
         ORG      0.858     0.913     0.884      1396
         PER      0.959     0.978     0.968       719

   micro avg      0.873     0.891     0.882      3525
   macro avg      0.850     0.866     0.858      3525
weighted avg      0.874     0.891     0.882      3525



**¿Por qué seqeval y no accuracy por token? ¿Qué tipo de entidad cuesta más?**

`accuracy` por token sería engañosamente alta e inútil aquí por dos razones: (1) la gran mayoría
de los tokens en un corpus NER llevan la etiqueta `O` (fuera de cualquier entidad), así que un
modelo que prediga `O` para todo ya obtendría accuracy altísima sin detectar ninguna entidad real
— el mismo problema de clases desbalanceadas que en B.4, pero más extremo; (2) accuracy por token
no entiende el esquema BIO: una entidad de varias palabras (`B-ORG I-ORG I-ORG`) solo cuenta como
correctamente detectada si **todas** sus subpalabras y fronteras (inicio `B-` y continuación `I-`)
coinciden, porque lo que importa en una aplicación real es extraer la entidad completa, no acertar
tokens sueltos. `seqeval` evalúa precisamente a **nivel de entidad completa**: una entidad predicha
solo cuenta como acierto si su tipo y su extensión exacta (todas las palabras que la componen)
coinciden con la entidad real, lo cual es la métrica que realmente importa para una aplicación de
extracción de información. En cuanto al tipo de entidad más difícil, típicamente es **MISC**
(miscelánea): a diferencia de PER, ORG y LOC que tienen patrones léxicos y contextuales bastante
consistentes (nombres propios de persona, sufijos corporativos, topónimos conocidos), MISC agrupa
una categoría heterogénea (nacionalidades, eventos, obras, adjetivos derivados de nombres propios)
sin un patrón superficial único, lo que la hace más ambigua y con menor F1 que las otras tres
clases.

### C.5 · Uso del modelo afinado — extraer entidades del corpus chiapaneco

In [160]:
from transformers import pipeline

extractor_entidades = pipeline(
    'ner', model=model, tokenizer=tok, aggregation_strategy='simple',
    device=0 if torch.cuda.is_available() else -1
)

docs_prueba = ['d07', 'd03', 'd09']   # mencionan instituciones, lugares y nombres propios

for doc_id in docs_prueba:
    texto = crudo[doc_id]
    print(f'\n— {doc_id}: {titulos[doc_id]} —')
    print(f'  Texto: {texto}')
    entidades = extractor_entidades(texto)
    if not entidades:
        print('  (sin entidades detectadas)')
    for ent in entidades:
        print(f'  → {ent["entity_group"]:<5} "{ent["word"]}"  (score={ent["score"]:.3f})')


— d07: UPCh inaugura laboratorio de IA —
  Texto: La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx .
  → ORG   "Universidad Politecnica de Chiapas"  (score=0.985)
  → MISC  "G"  (score=0.936)
  → MISC  "h"  (score=0.567)
  → MISC  "##ttps : /"  (score=0.632)
  → ORG   "upchiapas"  (score=0.479)

— d03: Cafe de Chiapas rompe record de exportacion —
  Texto: El cafe de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia. Los productores de la Sierra celebran precios al alza.
  → LOC   "Chiapas"  (score=0.996)
  → LOC   "Europa"  (score=0.996)
  → LOC   "Asia"  (score=0.994)
  → LOC   "Sierra"  (score=0.982)

— d09: San Cristobal, destino cultural —
  Texto: San Cristobal de las Casas se consolida como destino cultural: sus mercados, iglesias y cafeterias atraen a viajeros de 

**Comentario esperado:** d07 (UPCh inaugura laboratorio de IA) debería producir una entidad
`ORG` para "Universidad Politécnica de Chiapas"; d03 (café de Chiapas) debería producir `LOC` para
"Chiapas", "Europa" y "Asia"; d09 (San Cristóbal) debería producir `LOC` para "San Cristóbal de las
Casas". El resultado real confirma esto: las tres entidades esperadas aparecen con scores altos
(0.98–0.99), consistente con que CoNLL-2002 entrena bien sobre topónimos y organizaciones, aunque
el corpus de entrenamiento sea de noticias españolas de los 90 y los nombres aquí sean mexicanos —
el domain shift afecta poco a entidades bien formadas.

Donde sí aparece un problema real es en d07: el pipeline fragmenta la URL `https://upchiapas.edu.mx`
en subpalabras sueltas (`G`, `h`, `##ttps : /`) y las etiqueta como `MISC` con scores bajos
(0.48–0.94). Esto no es domain shift semántico, es un problema de tokenización: el tokenizer de
BETO parte la URL en piezas que no corresponden a ninguna entidad lingüística real, y como el
modelo nunca vio URLs durante el entrenamiento (CoNLL-2002 es texto periodístico de los 90, sin
hipervínculos), no tiene ninguna señal de contexto para decidir "esto no es una entidad". El score
bajo de esas detecciones (0.48–0.63, contra 0.98+ de las entidades reales) sí es una señal útil:
en un pipeline de producción, filtrar predicciones con `score < 0.7` habría eliminado este ruido
sin perder ninguna entidad verdadera de las tres pruebas.

Liberar memoria al terminar.

In [161]:
del model, trainer, extractor_entidades, conll_tok
liberar_memoria()

VRAM en uso: 0.02 GB


## Síntesis

Las tres partes usaron el mismo BERT preentrenado y solo cambiaron la cabeza y los datos: cabeza
siamesa con sus qrels (A), `[CLS]` + lineal con un dataset de sentimiento real (B), y una etiqueta
por token con CoNLL-2002 (C). Cada modelo afinado se aplicó sobre su propio corpus, y entre
entrenamientos se liberó memoria para sostener la sesión en una T4. Ese paradigma —preentrenar una
vez, adaptar barato— es la base de los sistemas RAG de la Unidad 3.

